# NLP de Comentarios TikTok / Instagram
## Que dice mi audiencia? Limpieza, sentimiento, topics y frases frecuentes

Analisis de texto aplicado a comentarios reales de redes sociales.  
Proyecto de [@aroaxinping](https://tiktok.com/@aroaxinping) — estudiante de Data Science.

---
## 0. Configuracion del entorno

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import re
import string
from collections import Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from wordcloud import WordCloud
import emoji

# Descargar recursos NLTK
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# --- Tema oscuro personalizado ---
BG_DARK   = '#0f0f0f'
ORANGE    = '#e85d04'
BLUE      = '#6a9ad4'
WHITE     = '#f0f0f0'
GRAY      = '#888888'

mpl.rcParams.update({
    'figure.facecolor': BG_DARK,
    'axes.facecolor':   BG_DARK,
    'axes.edgecolor':   GRAY,
    'axes.labelcolor':  WHITE,
    'text.color':       WHITE,
    'xtick.color':      GRAY,
    'ytick.color':      GRAY,
    'grid.color':       '#2a2a2a',
    'grid.alpha':       0.6,
    'figure.figsize':   (12, 5),
    'font.size':        11,
    'axes.grid':        True,
})

print("Entorno listo.")

---
## 1. Datos

Intentamos cargar comentarios reales desde `data/processed/comentarios.csv`.  
Si no existen, generamos un dataset sintetico de ~500 comentarios realistas.

In [ ]:
# -----------------------------------------------------------------------
# Intentar cargar datos reales procesados
# Si no existen, generar sinteticos
# -----------------------------------------------------------------------
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path

DATA_PATH = Path("../data/processed/comentarios.csv")
SYNTHETIC = False

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, parse_dates=["fecha"])
    print(f"Datos reales cargados: {len(df)} comentarios")
else:
    print("No hay datos reales. Generando dataset sintetico...")
    # Importar generador
    from fetch_comments import generate_synthetic_comments
    df = generate_synthetic_comments(500)
    SYNTHETIC = True
    print(f"Dataset sintetico: {len(df)} comentarios")

print(f"\nColumnas: {df.columns.tolist()}")
print(f"Fuentes: {df['fuente'].value_counts().to_dict()}")
df.head()

---
## 2. Limpieza de texto

> **Pregunta:** Como limpiar comentarios de redes sociales para analisis?

Los comentarios de TikTok/Instagram son ruidosos: emojis, URLs, menciones, mayusculas aleatorias, repeticiones de letras... Necesitamos un pipeline de limpieza antes de cualquier analisis.

In [ ]:
# -----------------------------------------------------------------------
# Pipeline de limpieza de texto
# -----------------------------------------------------------------------

# Stop words en espanol
STOP_WORDS = set(stopwords.words('spanish'))
# Anadir stop words custom para redes sociales
STOP_WORDS.update(['q', 'xq', 'xd', 'jaja', 'jajaja', 'jajajaja', 'pls', 'plz',
                   'porfa', 'porfavor', 'si', 'no', 'ya', 'like', 'for', 'follow'])


def extract_emojis(text: str) -> list:
    """Extrae todos los emojis de un texto."""
    return [c for c in text if c in emoji.EMOJI_DATA]


def clean_text(text: str) -> str:
    """Pipeline de limpieza para comentarios de redes sociales."""
    if not isinstance(text, str):
        return ""

    # 1. Extraer emojis antes de limpiar (los guardamos aparte)
    text_clean = emoji.replace_emoji(text, replace='')

    # 2. Eliminar URLs
    text_clean = re.sub(r'http\S+|www\.\S+', '', text_clean)

    # 3. Eliminar menciones (@usuario)
    text_clean = re.sub(r'@\w+', '', text_clean)

    # 4. Eliminar hashtags
    text_clean = re.sub(r'#\w+', '', text_clean)

    # 5. Lowercase
    text_clean = text_clean.lower()

    # 6. Eliminar puntuacion
    text_clean = text_clean.translate(str.maketrans('', '', string.punctuation))

    # 7. Reducir letras repetidas (muuuuy -> muy, increibleee -> increible)
    text_clean = re.sub(r'(.)\1{2,}', r'\1', text_clean)

    # 8. Eliminar espacios extra
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()

    return text_clean


def tokenize_and_filter(text: str) -> list:
    """Tokeniza y elimina stop words."""
    tokens = word_tokenize(text, language='spanish')
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]


# Aplicar pipeline
df['emojis'] = df['texto'].apply(extract_emojis)
df['texto_limpio'] = df['texto'].apply(clean_text)
df['tokens'] = df['texto_limpio'].apply(tokenize_and_filter)
df['n_palabras'] = df['tokens'].apply(len)
df['n_emojis'] = df['emojis'].apply(len)

# Mostrar antes vs despues
print("Ejemplo de limpieza:\n")
for i in range(5):
    print(f"  Original:  {df.iloc[i]['texto']}")
    print(f"  Limpio:    {df.iloc[i]['texto_limpio']}")
    print(f"  Tokens:    {df.iloc[i]['tokens']}")
    print()

---
## 3. Analisis exploratorio

> **Pregunta:** De que habla mi audiencia?

In [ ]:
# -----------------------------------------------------------------------
# 3a. Frecuencia de palabras
# -----------------------------------------------------------------------
all_tokens = [token for tokens in df['tokens'] for token in tokens]
word_freq = Counter(all_tokens)

top_25 = word_freq.most_common(25)
words, counts = zip(*top_25)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(words)), counts, color=ORANGE, alpha=0.85)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Frecuencia')
ax.set_title('Top 25 palabras mas frecuentes en comentarios', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# 3b. Wordcloud (fondo oscuro)
# -----------------------------------------------------------------------
text_for_cloud = ' '.join(all_tokens)

wc = WordCloud(
    width=1200,
    height=600,
    background_color=BG_DARK,
    colormap='Oranges',
    max_words=100,
    max_font_size=120,
    random_state=42,
    collocations=False,
).generate(text_for_cloud)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Wordcloud de comentarios', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# 3c. Distribucion de longitud de comentarios
# -----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Longitud en palabras
axes[0].hist(df['n_palabras'], bins=20, color=ORANGE, alpha=0.8, edgecolor=BG_DARK)
axes[0].set_xlabel('Numero de palabras')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribucion: longitud de comentarios (palabras)', fontweight='bold')
axes[0].axvline(df['n_palabras'].median(), color=BLUE, linestyle='--', label=f"Mediana: {df['n_palabras'].median():.0f}")
axes[0].legend()

# Longitud en caracteres
char_lengths = df['texto'].str.len()
axes[1].hist(char_lengths, bins=30, color=BLUE, alpha=0.8, edgecolor=BG_DARK)
axes[1].set_xlabel('Numero de caracteres')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion: longitud de comentarios (caracteres)', fontweight='bold')
axes[1].axvline(char_lengths.median(), color=ORANGE, linestyle='--', label=f"Mediana: {char_lengths.median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Longitud media: {df['n_palabras'].mean():.1f} palabras, {char_lengths.mean():.0f} caracteres")
print(f"Comentarios vacios (0 tokens tras limpieza): {(df['n_palabras'] == 0).sum()}")

In [ ]:
# -----------------------------------------------------------------------
# 3d. Analisis de emojis
# -----------------------------------------------------------------------
all_emojis = [e for emojis_list in df['emojis'] for e in emojis_list]
emoji_freq = Counter(all_emojis)

if emoji_freq:
    top_emojis = emoji_freq.most_common(15)
    emojis_chars, emojis_counts = zip(*top_emojis)

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(range(len(emojis_chars)), emojis_counts, color=ORANGE, alpha=0.85)
    ax.set_xticks(range(len(emojis_chars)))
    ax.set_xticklabels(emojis_chars, fontsize=20)
    ax.set_ylabel('Frecuencia')
    ax.set_title('Top 15 emojis mas usados', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\nTotal emojis encontrados: {len(all_emojis)}")
    print(f"Emojis unicos: {len(emoji_freq)}")
    print(f"Comentarios con emojis: {(df['n_emojis'] > 0).sum()} ({(df['n_emojis'] > 0).mean()*100:.0f}%)")
else:
    print("No se encontraron emojis en los comentarios.")

---
## 4. Analisis de sentimiento

> **Pregunta:** Mi audiencia es positiva, negativa o neutra?

Usamos un enfoque basado en lexico: un diccionario de palabras positivas y negativas en espanol.  
No es perfecto, pero funciona razonablemente para textos cortos de redes sociales.

In [ ]:
# -----------------------------------------------------------------------
# Lexico de sentimiento en espanol (simplificado pero funcional)
# -----------------------------------------------------------------------

POSITIVE_WORDS = {
    'encanta', 'increible', 'genial', 'brutal', 'mejor', 'crack', 'top',
    'inspiracion', 'inspiras', 'motivas', 'motivacion', 'gracias', 'grande',
    'love', 'fire', 'wow', 'guau', 'excelente', 'perfecto', 'bueno', 'buena',
    'bonito', 'bonita', 'hermoso', 'hermosa', 'maravilloso', 'fantastico',
    'recomiendo', 'calidad', 'aprendo', 'favorito', 'favorita', 'animo',
    'ejemplo', 'alegra', 'limpio', 'goals', 'bien', 'contento', 'feliz',
    'super', 'impresionante', 'espectacular', 'magnifico', 'brillante',
}

NEGATIVE_WORDS = {
    'mal', 'malo', 'mala', 'peor', 'horrible', 'feo', 'fea', 'aburrido',
    'aburrida', 'lento', 'bajo', 'baja', 'corto', 'corta', 'falto',
    'entendi', 'asco', 'basura', 'odio', 'spam', 'fake', 'fraude',
    'robo', 'mentira', 'decepcion', 'error', 'problema', 'dificil',
}


def sentiment_score(tokens: list) -> float:
    """Calcula un score de sentimiento simple basado en lexico."""
    if not tokens:
        return 0.0
    pos = sum(1 for t in tokens if t in POSITIVE_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIVE_WORDS)
    total = len(tokens)
    return (pos - neg) / total


def sentiment_label(score: float) -> str:
    if score > 0.05:
        return 'positivo'
    elif score < -0.05:
        return 'negativo'
    else:
        return 'neutro'


df['sentimiento_score'] = df['tokens'].apply(sentiment_score)
df['sentimiento'] = df['sentimiento_score'].apply(sentiment_label)

print("Distribucion de sentimiento:")
print(df['sentimiento'].value_counts())
print(f"\nScore medio: {df['sentimiento_score'].mean():.3f}")

In [ ]:
# -----------------------------------------------------------------------
# 4b. Visualizacion de sentimiento
# -----------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribucion general
colors_sent = {'positivo': '#2ecc71', 'neutro': GRAY, 'negativo': '#e74c3c'}
sent_counts = df['sentimiento'].value_counts()
bars = axes[0].bar(sent_counts.index, sent_counts.values,
                   color=[colors_sent[s] for s in sent_counts.index], alpha=0.85)
axes[0].set_title('Distribucion de sentimiento', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Numero de comentarios')
for bar, count in zip(bars, sent_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 str(count), ha='center', color=WHITE, fontweight='bold')

# Score distribution
axes[1].hist(df['sentimiento_score'], bins=30, color=ORANGE, alpha=0.8, edgecolor=BG_DARK)
axes[1].axvline(0, color=WHITE, linestyle='--', alpha=0.5)
axes[1].set_xlabel('Score de sentimiento')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion del score de sentimiento', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# 4c. Sentimiento por tema de video
# -----------------------------------------------------------------------
if 'video_topic' in df.columns:
    sent_by_topic = df.groupby('video_topic')['sentimiento_score'].agg(['mean', 'count'])
    sent_by_topic = sent_by_topic[sent_by_topic['count'] >= 5].sort_values('mean', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 6))
    colors_bar = [('#2ecc71' if v > 0 else '#e74c3c') for v in sent_by_topic['mean']]
    ax.barh(range(len(sent_by_topic)), sent_by_topic['mean'], color=colors_bar, alpha=0.85)
    ax.set_yticks(range(len(sent_by_topic)))
    ax.set_yticklabels(sent_by_topic.index, fontsize=10)
    ax.set_xlabel('Score medio de sentimiento')
    ax.set_title('Sentimiento medio por tema de video', fontsize=14, fontweight='bold')
    ax.axvline(0, color=WHITE, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Columna video_topic no disponible (solo en datos sinteticos).")

In [ ]:
# -----------------------------------------------------------------------
# 4d. Comentarios mas positivos y mas negativos
# -----------------------------------------------------------------------
print("=== TOP 5 comentarios MAS POSITIVOS ===\n")
top_pos = df.nlargest(5, 'sentimiento_score')
for _, row in top_pos.iterrows():
    print(f"  [{row['sentimiento_score']:.2f}] {row['texto']}")

print("\n=== TOP 5 comentarios MAS NEGATIVOS ===\n")
top_neg = df.nsmallest(5, 'sentimiento_score')
for _, row in top_neg.iterrows():
    print(f"  [{row['sentimiento_score']:.2f}] {row['texto']}")

---
## 5. Topic modeling (basico)

> **Pregunta:** Se pueden detectar temas recurrentes en los comentarios?

Usamos TF-IDF + K-Means para agrupar comentarios en clusters tematicos.  
No es LDA ni transformers, pero para comentarios cortos funciona sorprendentemente bien.

In [ ]:
# -----------------------------------------------------------------------
# 5a. TF-IDF vectorizacion
# -----------------------------------------------------------------------
# Filtrar comentarios con al menos 2 tokens
df_topics = df[df['n_palabras'] >= 2].copy()
print(f"Comentarios con >= 2 tokens: {len(df_topics)} / {len(df)}")

# TF-IDF
tfidf = TfidfVectorizer(
    max_features=500,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
)
tfidf_matrix = tfidf.fit_transform(df_topics['texto_limpio'])
print(f"Matriz TF-IDF: {tfidf_matrix.shape}")

# Feature names
feature_names = tfidf.get_feature_names_out()
print(f"Vocabulario: {len(feature_names)} terminos")

In [ ]:
# -----------------------------------------------------------------------
# 5b. K-Means clustering
# -----------------------------------------------------------------------
N_CLUSTERS = 5

km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df_topics['cluster'] = km.fit_predict(tfidf_matrix)

# Palabras mas representativas de cada cluster
print("=== Temas detectados (top palabras por cluster) ===\n")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]

cluster_labels = {}
for i in range(N_CLUSTERS):
    top_words = [feature_names[ind] for ind in order_centroids[i, :8]]
    cluster_labels[i] = ', '.join(top_words[:3])
    print(f"  Cluster {i}: {', '.join(top_words)}")

print(f"\nComentarios por cluster:")
print(df_topics['cluster'].value_counts().sort_index())

In [ ]:
# -----------------------------------------------------------------------
# 5c. Visualizacion de clusters
# -----------------------------------------------------------------------
from sklearn.decomposition import PCA

# Reducir a 2D para visualizar
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(tfidf_matrix.toarray())

cluster_colors = [ORANGE, BLUE, '#2ecc71', '#e74c3c', '#9b59b6']

fig, ax = plt.subplots(figsize=(12, 8))
for i in range(N_CLUSTERS):
    mask = df_topics['cluster'] == i
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=cluster_colors[i % len(cluster_colors)],
               alpha=0.6, s=30, label=f'Cluster {i}: {cluster_labels[i]}')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Clusters de comentarios (TF-IDF + K-Means + PCA)', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=9, framealpha=0.3)
ax.grid(False)
plt.tight_layout()
plt.show()

---
## 6. N-gramas y frases frecuentes

> **Pregunta:** Cuales son las frases mas repetidas?

In [ ]:
# -----------------------------------------------------------------------
# 6a. Bigramas y trigramas
# -----------------------------------------------------------------------
from nltk import ngrams

def get_ngrams(tokens_series, n):
    """Extrae n-gramas de una serie de listas de tokens."""
    all_ngrams = []
    for tokens in tokens_series:
        if len(tokens) >= n:
            all_ngrams.extend(list(ngrams(tokens, n)))
    return Counter(all_ngrams)

bigrams = get_ngrams(df['tokens'], 2)
trigrams = get_ngrams(df['tokens'], 3)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top bigramas
top_bi = bigrams.most_common(15)
if top_bi:
    bi_labels, bi_counts = zip(*top_bi)
    bi_labels = [' '.join(b) for b in bi_labels]
    axes[0].barh(range(len(bi_labels)), bi_counts, color=ORANGE, alpha=0.85)
    axes[0].set_yticks(range(len(bi_labels)))
    axes[0].set_yticklabels(bi_labels, fontsize=9)
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Frecuencia')
    axes[0].set_title('Top 15 bigramas', fontsize=14, fontweight='bold')

# Top trigramas
top_tri = trigrams.most_common(15)
if top_tri:
    tri_labels, tri_counts = zip(*top_tri)
    tri_labels = [' '.join(t) for t in tri_labels]
    axes[1].barh(range(len(tri_labels)), tri_counts, color=BLUE, alpha=0.85)
    axes[1].set_yticks(range(len(tri_labels)))
    axes[1].set_yticklabels(tri_labels, fontsize=9)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Frecuencia')
    axes[1].set_title('Top 15 trigramas', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# 6b. Frases unicas por cluster/topic
# -----------------------------------------------------------------------
if 'cluster' in df_topics.columns:
    print("=== Bigramas mas frecuentes por cluster ===\n")
    for c in range(N_CLUSTERS):
        cluster_tokens = df_topics[df_topics['cluster'] == c]['tokens']
        cluster_bigrams = get_ngrams(cluster_tokens, 2)
        top_5 = cluster_bigrams.most_common(5)
        phrases = [' '.join(b) for b, _ in top_5]
        print(f"  Cluster {c} ({cluster_labels[c]}):")
        for phrase, count in top_5:
            print(f"    - {' '.join(phrase)} ({count})")
        print()

---
## 7. Sintesis y conclusiones

In [ ]:
# -----------------------------------------------------------------------
# Resumen automatico del analisis
# -----------------------------------------------------------------------

print("=" * 60)
print("  RESUMEN DEL ANALISIS NLP")
print("=" * 60)

print(f"\n  Total comentarios analizados: {len(df)}")
print(f"  Fuente: {'sintetico' if SYNTHETIC else 'datos reales'}")

print(f"\n  --- Texto ---")
print(f"  Longitud media: {df['n_palabras'].mean():.1f} palabras")
print(f"  Vocabulario unico: {len(set(all_tokens))} palabras")
print(f"  Palabra mas frecuente: '{word_freq.most_common(1)[0][0]}' ({word_freq.most_common(1)[0][1]} veces)")

print(f"\n  --- Emojis ---")
print(f"  Comentarios con emojis: {(df['n_emojis'] > 0).sum()} ({(df['n_emojis'] > 0).mean()*100:.0f}%)")
if emoji_freq:
    print(f"  Emoji mas usado: {emoji_freq.most_common(1)[0][0]} ({emoji_freq.most_common(1)[0][1]} veces)")

print(f"\n  --- Sentimiento ---")
sent_pct = df['sentimiento'].value_counts(normalize=True) * 100
for label in ['positivo', 'neutro', 'negativo']:
    if label in sent_pct:
        print(f"  {label.capitalize()}: {sent_pct[label]:.1f}%")

print(f"\n  --- Topics (K-Means) ---")
print(f"  Clusters detectados: {N_CLUSTERS}")
for i in range(N_CLUSTERS):
    n = (df_topics['cluster'] == i).sum()
    print(f"  Cluster {i}: {cluster_labels[i]} ({n} comentarios)")

print(f"\n  --- N-gramas ---")
if top_bi:
    print(f"  Bigrama mas frecuente: '{' '.join(top_bi[0][0])}' ({top_bi[0][1]} veces)")
if top_tri:
    print(f"  Trigrama mas frecuente: '{' '.join(top_tri[0][0])}' ({top_tri[0][1]} veces)")

print("\n" + "=" * 60)